# Rule-check model: GPU training on Colab (Phase 3)

A thin wrapper over the repository's versioned scripts (`ml/`); it adds no modelling code. On the
**full official ContractNLI splits** it trains `deberta_v3_base` (primary), `modernbert_base`
(backbone ablation) and `deberta_v3_base_unseen4` (4 hypotheses held out of training), evaluates them
against the baselines with DEV-only choices, exports ONNX fp32/int8 with parity checks, measures CPU
latency and int8/fp32 TEST accuracy on this runtime, and packs a **verifiable results bundle**.

Read `ml/notebooks/README.md` first. In short:
- Runtime -> Change runtime type -> **T4 GPU**.
- Source without secrets: upload a `git bundle` of the reviewed commit to Drive (**Option A**,
  recommended), or store a read-only fine-grained GitHub token as the **Colab secret** `GITHUB_TOKEN`
  (Option B). Never paste a token or API key into a cell.
- `RUN_LOCATION = "local"` (default): runs, checkpoints, exports and the results bundle are written to
  this runtime's disk. Drive (`DRIVE_ROOT`) provides the source bundle, the prepared data and the
  baselines, and receives only space-checked, verified backups in new folders; nothing on Drive is
  ever deleted. Local files live as long as this runtime (VM): a browser reconnect to the same
  runtime keeps them and a rerun resumes from the local checkpoint, but when the runtime is deleted,
  reset or expires they are gone and an unfinished run starts again from step 0. Every command's output is shown live and saved under `RUNS/logs/`.

No cell prints, logs or stores a token or API key.

## 1. GPU and Drive

In [ ]:
# ruff: noqa: S101, S603, S607, T201, E402  (notebook: shell calls, prints and asserts are intended)
import json
import pathlib
import shutil
import subprocess

subprocess.run(["nvidia-smi"], check=True)
from google.colab import drive

drive.mount("/content/drive")
# Drive: source bundle, prepared data and baselines (read), and explicit space-checked backups.
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/compliance-runs")
# "local" (default): runs, checkpoints and exports on this runtime's disk; they survive a
# browser reconnect to the same runtime, not its deletion, reset or expiry.
# "drive": everything under DRIVE_ROOT as before; each checkpoint rotation needs ~2.2 GB and deleted
# checkpoints are expected to stay in Drive Trash, counting against the quota (see README).
RUN_LOCATION = "local"
assert RUN_LOCATION in ("local", "drive"), RUN_LOCATION
RUNS = DRIVE_ROOT if RUN_LOCATION == "drive" else pathlib.Path("/content/compliance-runs")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path("/content/compliance")
for where, path in (("local disk", "/content"), ("Drive", DRIVE_ROOT)):
    usage = shutil.disk_usage(path)
    print(f"{where}: {usage.free / 1024**3:.1f} GiB free of {usage.total / 1024**3:.1f} GiB")
print("Drive root:", DRIVE_ROOT, "| runs:", RUNS, f"({RUN_LOCATION})")

In [ ]:
# Helpers: definitions only (exercised by ml/tests/test_notebook.py). They use RUNS, DRIVE_ROOT,
# RUN_LOCATION and, once the environment cell has run, PY, assert_cuda and EXPECTED_COMMIT.
import contextlib
import datetime
import hashlib
import os
import pathlib
import shutil
import signal
import subprocess
import threading

GIB = 1024**3
# After an interrupt: seconds to wait after SIGINT, then after SIGTERM, before SIGKILL.
STOP_GRACE_S = (30.0, 10.0)
# One training run: checkpoints/last + staging while promoting (2 x 2.2 GB), best, reserve.
TRAIN_SPACE_GIB = 10


def _stamp():
    return datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%S%fZ")


def _say(message):
    """Print, never raising: used while stopping a child, when the output itself may be broken."""
    with contextlib.suppress(Exception):
        print(message, flush=True)


def _stop_child(proc):
    """SIGINT, then SIGTERM, then SIGKILL to the child's process group until it has exited."""
    steps = (
        (signal.SIGINT, STOP_GRACE_S[0]),
        (signal.SIGTERM, STOP_GRACE_S[1]),
        (signal.SIGKILL, None),
    )
    for sig, timeout in steps:
        if proc.poll() is not None:
            break
        try:
            os.killpg(proc.pid, sig)
        except ProcessLookupError:
            break
        try:
            proc.wait(timeout=timeout)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            continue  # a repeated interrupt escalates instead of abandoning the child
    proc.wait()
    with contextlib.suppress(ProcessLookupError):
        os.killpg(proc.pid, signal.SIGKILL)  # anything the child left behind in its group


def stream(cmd, name, env=None):
    """Run cmd with its stdout and stderr shown live in the cell and appended to RUNS/logs/.

    Raises CalledProcessError on a non-zero exit. On an interrupt (the stop button), or if showing
    or logging the output fails (RuntimeError, chained), the child is stopped with _stop_child and
    waited for before the error propagates, so no orphan keeps writing. Only the command line and
    the child's output are shown or logged, never the environment (secrets reach a child only
    through env).
    """
    cmd = [str(c) for c in cmd]
    log_dir = pathlib.Path(RUNS) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{_stamp()}-{name}.log"
    print("$", " ".join(cmd), flush=True)
    print("  log:", log_path, flush=True)
    proc = subprocess.Popen(
        cmd,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})},
        text=True,
        encoding="utf-8",
        errors="replace",
        start_new_session=True,  # an interrupt reaches the child only through _stop_child
    )

    relay_error = []

    def pump():
        try:
            with log_path.open("a", encoding="utf-8") as log:
                for line in proc.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
        except Exception as exc:  # reported by the main thread
            relay_error.append(exc)
            with contextlib.suppress(Exception):
                for _ in proc.stdout:  # keep draining: the child must never block on a full pipe
                    pass

    def relay_failed():
        return RuntimeError(f"showing/logging the output of {name} failed; log: {log_path}")

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    try:
        while True:
            if relay_error:
                raise relay_failed() from relay_error[0]
            try:
                code = proc.wait(timeout=1)
                break
            except subprocess.TimeoutExpired:
                continue
    except BaseException as exc:
        _say(f"\nstopping {name} (pid {proc.pid}) after {type(exc).__name__} ...")
        _stop_child(proc)
        reader.join(timeout=30)
        _say(f"{name} stopped (exit {proc.returncode}); log: {log_path}")
        raise
    reader.join(timeout=30)
    if code == 0 and relay_error:
        raise relay_failed() from relay_error[0]
    if code != 0:
        print(f"FAILED: {name} exited with {code}; output in {log_path}", flush=True)
        raise subprocess.CalledProcessError(code, cmd)
    return log_path


def run(*args, **env):
    """A repository module with the project interpreter (see stream)."""
    return stream([PY, "-m", *args], args[0], {"PYTHONPATH": "", **env})


def free_gib(path):
    return shutil.disk_usage(path).free / GIB


def require_space(path, gib, what):
    """Stop (deleting nothing) unless path's filesystem has at least gib GiB free."""
    free = free_gib(path)
    if free < gib:
        raise RuntimeError(
            f"{what}: {free:.1f} GiB free at {path}, {gib} GiB needed; free space deliberately "
            "(nothing is deleted automatically)"
        )
    print(f"{what}: {free:.1f} GiB free at {path} ({gib} GiB needed)")


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        while chunk := fh.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def _backup_files(src, exclude):
    """(base, relative files) to back up; symlinks and special files are refused up front."""
    if src.is_symlink() or not src.exists():
        raise RuntimeError(f"cannot back up {src}: missing or a symlink")
    if src.is_file():
        return src.parent, [pathlib.PurePosixPath(src.name)]
    excluded = [pathlib.PurePosixPath(e) for e in exclude]
    files = []
    for path in sorted(src.rglob("*")):
        rel = pathlib.PurePosixPath(path.relative_to(src).as_posix())
        if any(rel == e or e in rel.parents for e in excluded):
            continue
        if path.is_symlink() or not (path.is_file() or path.is_dir()):
            raise RuntimeError(f"cannot back up {path}: not a regular file or directory")
        if path.is_file():
            files.append(rel)
    return src, files


def backup_to_drive(src, label, exclude=(), reserve_gib=1.0):
    """Copy a finished output to a NEW folder DRIVE_ROOT/backups/<label>-<UTC time>, verified.

    Copies nothing unless Drive has room for it plus reserve_gib (Trash counts against the quota).
    Never deletes, overwrites or reuses anything on Drive. The read-back check verifies the write
    through the mount; the upload to Google completes asynchronously, so confirm the folder on
    drive.google.com before relying on it. Returns the backup folder, or None when skipped.
    """
    src = pathlib.Path(src)
    if pathlib.Path(RUNS) == pathlib.Path(DRIVE_ROOT):
        print(f"backup skipped: {src} is already on Drive")
        return None
    base, files = _backup_files(src, exclude)
    size = sum((base / f).stat().st_size for f in files)
    need, free = size / GIB + reserve_gib, free_gib(DRIVE_ROOT)
    if free < need:
        print(
            f"backup of {src} SKIPPED: Drive has {free:.2f} GiB free, {need:.2f} GiB needed. Nothing "
            "was copied or deleted; download it (file browser) or free Drive space and call "
            "backup_to_drive again."
        )
        return None
    dest = pathlib.Path(DRIVE_ROOT) / "backups" / f"{label}-{_stamp()}"
    dest.mkdir(parents=True)  # a new folder every time
    sums = []
    for rel in files:
        target = dest / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(base / rel, target)
        digest = _sha256(base / rel)
        if _sha256(target) != digest:
            raise RuntimeError(f"backup verification failed for {rel}; the partial copy is {dest}")
        sums.append(f"{digest}  {rel}\n")
    (dest / "SHA256SUMS").write_text("".join(sums))
    print(f"backed up {len(files)} files ({size / GIB:.2f} GiB) to {dest} (read-back verified)")
    return dest


def train(name, config):
    """One training run in RUNS/<name>; in local mode a verified Drive backup once it completes
    (weights of the selected epoch, records and predictions; not the 2.2 GB checkpoints/last)."""
    assert_cuda()
    require_space(RUNS, TRAIN_SPACE_GIB, f"training {name}")
    run("ml.train.train", "--config", config, "--run-dir", str(pathlib.Path(RUNS) / name))
    if RUN_LOCATION == "local":
        backup_to_drive(
            pathlib.Path(RUNS) / name,
            f"{name}-{EXPECTED_COMMIT[:12]}",
            exclude=("checkpoints/last",),
        )

## 2. Source code (choose ONE option)

**Option A (recommended, no token):** on the development machine, after the reviewed commit exists:
`git bundle create compliance.bundle <branch>` and upload the file to `MyDrive/compliance-runs/`.
**Option B:** add a fine-grained, read-only (Contents: read) GitHub token for this one repository as the
Colab secret `GITHUB_TOKEN`, and set `USE_OPTION_B_GITHUB = True`.
In both cases set `EXPECTED_COMMIT` to the full commit hash to train; the cell refuses anything else.

In [ ]:
EXPECTED_COMMIT = ""  # full 40-character commit hash (required)
BUNDLE = DRIVE_ROOT / "compliance.bundle"  # Option A
USE_OPTION_B_GITHUB = False
GITHUB_REPO = "JCHETAN26/compliance"

import tempfile

# The pipeline itself rewrites ml/eval/results/ (data_report.json is tracked; evaluation adds
# new files), so a rerun in the same runtime excludes ONLY that generated subtree. Any other
# changed or untracked file still refuses to train.
GENERATED_RESULTS = "ml/eval/results"


def git(repo, *args):
    """A git command in repo; on failure raises RuntimeError with git's own message."""
    out = subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {out.stderr.strip()}")
    return out.stdout


def implementation_changes(repo):
    """`git status` lines for changed/untracked files outside the generated results tree."""
    return subprocess.run(
        [
            "git",
            "-C",
            str(repo),
            "status",
            "--porcelain",
            "--untracked-files=all",
            "--",
            ".",
            f":(exclude){GENERATED_RESULTS}",
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout


def has_commit(repo, commit):
    probe = ["git", "-C", str(repo), "cat-file", "-e", f"{commit}^{{commit}}"]
    return subprocess.run(probe, capture_output=True).returncode == 0


def clone_github(repo):
    # The token comes from a Colab secret and reaches git only through the environment of a
    # GIT_ASKPASS helper: never printed, never in a URL, never written to .git/config.
    from google.colab import userdata

    helper = pathlib.Path("/content/.git-askpass.sh")
    helper.write_text(
        '#!/bin/sh\ncase "$1" in Username*) echo x-access-token;; *) echo "$CLONE_TOKEN";; esac\n'
    )
    helper.chmod(0o700)
    env = {
        **os.environ,
        "GIT_ASKPASS": str(helper),
        "GIT_TERMINAL_PROMPT": "0",
        "CLONE_TOKEN": userdata.get("GITHUB_TOKEN"),
    }
    try:
        subprocess.run(
            ["git", "clone", "--quiet", f"https://github.com/{GITHUB_REPO}.git", str(repo)],
            check=True,
            env=env,
        )
    finally:
        helper.unlink(missing_ok=True)
        del env


def checkout_source(repo, commit, bundle, use_github):
    """Check out commit in repo and return HEAD; refuses rather than overwriting anything.

    A fresh runtime clones (Option A: the verified bundle; Option B: GitHub). In an existing
    checkout (same runtime) a missing commit is fetched from the verified bundle into FETCH_HEAD
    only (no branch update, reset or working-tree change), and the checkout happens only while the
    implementation is clean; git itself refuses to overwrite a changed generated file.
    """
    if not repo.exists():
        if use_github:
            clone_github(repo)
        else:
            # `git bundle verify` needs a repository: verify inside a fresh empty one, then clone.
            check = pathlib.Path(tempfile.mkdtemp(prefix="bundle-check-"))
            git(check, "init", "--quiet")
            git(check, "bundle", "verify", str(bundle))
            shutil.rmtree(check)
            subprocess.run(["git", "clone", "--quiet", str(bundle), str(repo)], check=True)
    elif not has_commit(repo, commit):
        if use_github:
            raise RuntimeError(
                f"{commit} is not in the existing checkout {repo}. Use Option A (upload the new "
                "bundle), or Runtime -> Disconnect and delete runtime and run all cells again."
            )
        git(repo, "bundle", "verify", str(bundle))
        refs = [
            line.split()[1] for line in git(repo, "bundle", "list-heads", str(bundle)).splitlines()
        ]
        git(repo, "fetch", "--quiet", str(bundle), *refs)  # FETCH_HEAD only
    if not has_commit(repo, commit):
        raise RuntimeError(f"{commit} is not in {bundle if not use_github else GITHUB_REPO}")
    dirty = implementation_changes(repo)
    if dirty:
        raise RuntimeError(f"implementation changes in {repo}; refusing to check out:\n{dirty}")
    git(repo, "checkout", "--quiet", commit)
    return git(repo, "rev-parse", "HEAD").strip()


assert len(EXPECTED_COMMIT) == 40, "set EXPECTED_COMMIT to the full commit hash"
head = checkout_source(REPO, EXPECTED_COMMIT, BUNDLE, USE_OPTION_B_GITHUB)
dirty = implementation_changes(REPO)
assert head == EXPECTED_COMMIT, head
assert not dirty, dirty
print("training source:", head, "(implementation clean; generated results excluded)")
os.chdir(REPO)

## 3. Environment: the locked dependencies + CUDA PyTorch of the same version

`uv sync --frozen` installs exactly `uv.lock` (Python 3.11, CPU torch 2.14.1); the torch wheel is then
replaced by the **CUDA 12.6 build of the same version** and the ML tool pins are added. All later
commands call `.venv/bin/python` directly (never `uv run` or `make`, which would re-sync the CPU
torch), and `assert_cuda()` re-checks the build right before every GPU step.

In [ ]:
stream(["pip", "install", "--quiet", "uv==0.8.22"], "install-uv")
stream(["uv", "sync", "--frozen"], "uv-sync")
stream(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall-package",
        "torch",
        "torch==2.14.1+cu126",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ],
    "cuda-torch",
)
stream(
    ["uv", "pip", "install", "--python", ".venv/bin/python", "-r", "ml/requirements-tools.txt"],
    "ml-tools",
)
PY = str(REPO / ".venv/bin/python")


def assert_cuda():
    out = subprocess.run(
        [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()
    assert out[0] == "2.14.1+cu126", f"wrong torch build {out[0]} (was the environment re-synced?)"
    assert out[1] == "True", "CUDA is not available"


assert_cuda()
print("torch 2.14.1+cu126 with CUDA ready")

## 4. Pinned artifacts (ContractNLI archive + BGE encoder; SHA-256 verified)

In [ ]:
run("compliance_ingestion.embeddings", "fetch", ARTIFACT_CACHE_DIR=".cache")
run("compliance_seed.fetch", ARTIFACT_CACHE_DIR=".cache")

## 5. Processed data (official splits, segments, candidate ranking) and classic baselines

Prepared once and kept on Drive, so every session and every run uses the **same `data_version`** (a
resumed run refuses a different one). The dense candidate ranking uses the pinned BGE encoder on GPU.
The majority and TF-IDF baselines run on CPU on the same data.

In [ ]:
DATA_DRIVE = DRIVE_ROOT / "data-full"  # canonical prepared data (same data_version every session)
DATA = REPO / ".cache/ml/data/full"
if DATA_DRIVE.exists():
    shutil.rmtree(DATA, ignore_errors=True)  # the local copy only
    shutil.copytree(DATA_DRIVE, DATA)
    print("restored data from Drive")
else:
    assert_cuda()
    run("ml.data.prepare", "--device", "cuda", "--threads", "2")
    data_gib = sum(p.stat().st_size for p in DATA.rglob("*") if p.is_file()) / GIB
    if free_gib(DRIVE_ROOT) >= data_gib + 1:
        shutil.copytree(DATA, DATA_DRIVE)
    else:
        print(
            "prepared data NOT saved to Drive (not enough space); a later session prepares it again"
        )
stream(
    [
        PY,
        "-c",
        "from pathlib import Path; from ml.data.store import load_dataset; "
        "load_dataset(Path('.cache/ml/data/full'))",
    ],
    "verify-data",
    {"PYTHONPATH": ""},
)  # verifies every file hash
manifest = json.loads((DATA / "manifest.json").read_text())
print("data_version", manifest["data_version"], manifest["counts"])
BASELINES = RUNS / "baselines"
BASELINES_DRIVE = DRIVE_ROOT / "baselines"
if BASELINES != BASELINES_DRIVE and not BASELINES.exists() and BASELINES_DRIVE.is_dir():
    shutil.copytree(BASELINES_DRIVE, BASELINES)  # read from Drive; Drive is left as it is
    print("copied baselines from Drive")
if not (BASELINES / "classic_baselines.json").exists():
    run("ml.baselines.classic", "--out", str(BASELINES))
    backup_to_drive(BASELINES, "baselines")

## 6. Training

Each run selects its checkpoint/epoch and label top-k on DEV only; TEST is predicted after that selection
is frozen. With `RUN_LOCATION = "local"` checkpoints stay on this runtime's disk: rerunning a cell in the
**same** runtime (also after a browser reconnect) resumes from `checkpoints/last`; once the runtime is
deleted, reset or expired, the run starts again from step 0. A completed run is backed up to
`DRIVE_ROOT/backups/` if Drive has room (otherwise it says so and copies nothing); the backup has no
`checkpoints/last` and is never restored automatically. A different data version, tokenizer or training code is refused on resume.

In [ ]:
train("deberta_v3_base", "ml/train/configs/deberta_v3_base.yaml")

In [ ]:
train("modernbert_base", "ml/train/configs/modernbert_base.yaml")

In [ ]:
train("deberta_v3_base_unseen4", "ml/train/configs/deberta_v3_base_unseen4.yaml")

## 7. Evaluation, export, CPU latency and quantized TEST accuracy

The serving backbone is chosen on DEV. Export writes ONNX fp32 + int8 with parity checks to `RUNS` (backed up to Drive at the end if it has room). The
benchmark measures this runtime's CPU (recorded in the result file). The int8/fp32 TEST accuracy run
is required for the phase and takes a while on Colab's CPU.

In [ ]:
def evaluate(*extra):
    run(
        "ml.eval.evaluate",
        "--run-dir",
        str(RUNS / "deberta_v3_base"),
        "--ablation-run-dir",
        str(RUNS / "modernbert_base"),
        "--unseen-run-dir",
        str(RUNS / "deberta_v3_base_unseen4"),
        "--baselines-dir",
        str(BASELINES),
        *extra,
    )


evaluate()
params = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
CHOSEN = pathlib.Path(params["run_dir"])
ARTIFACT = RUNS / f"artifact-{CHOSEN.name}"
assert_cuda()
run(
    "ml.export.export",
    "--run-dir",
    str(CHOSEN),
    "--params",
    "ml/eval/results/inference_params.json",
    "--model-eval",
    "ml/eval/results/model_eval.json",
    "--out",
    str(ARTIFACT),
)
THREADS = str(os.cpu_count() or 2)
run(
    "ml.export.bench",
    "--artifact-dir",
    str(ARTIFACT),
    "--run-dir",
    str(CHOSEN),
    "--label",
    "colab_cpu",
    "--threads",
    THREADS,
    "--out",
    "ml/eval/results/latency_colab_cpu.json",
)
run(
    "ml.export.eval_onnx",
    "--artifact-dir",
    str(ARTIFACT),
    "--run-dir",
    str(CHOSEN),
    "--split",
    "test",
    "--threads",
    THREADS,
    "--out",
    "ml/eval/results/quantization_eval.json",
)

## 8. Optional: the Anthropic zero-shot baseline here (real API calls, billed to your key)

Leave `RUN_LLM = False` to run it locally after import instead (`make baseline-llm`). If enabled, the key
is read from the Colab secret `ANTHROPIC_API_KEY` into the child process's environment only. The dry
runs print the prompt and a rough cost estimate; re-running resumes without paying twice.

In [ ]:
RUN_LLM = False
if RUN_LLM:
    from google.colab import userdata

    for split in ("dev", "test"):
        run("ml.baselines.llm", "--split", split, "--dry-run")
    key_env = {"ANTHROPIC_API_KEY": userdata.get("ANTHROPIC_API_KEY")}
    for split in ("dev", "test"):
        run("ml.baselines.llm", "--split", split, "--out", str(BASELINES), **key_env)
    del key_env

## 9. Final evaluation and artifact parameters

Re-evaluates with the measured latency (and the LLM results if present: the cascade tau is then
selected on DEV), and refreshes the artifact manifest so the served parameters and version match the
final DEV selections (weights unchanged).

In [ ]:
evaluate("--bench", "ml/eval/results/latency_colab_cpu.json")
final = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
assert final["run_dir"] == params["run_dir"], "the DEV backbone choice changed; re-run step 7"
run(
    "ml.export.export",
    "--run-dir",
    str(CHOSEN),
    "--params",
    "ml/eval/results/inference_params.json",
    "--model-eval",
    "ml/eval/results/model_eval.json",
    "--out",
    str(ARTIFACT),
    "--refresh-manifest",
)

## 10. Results bundle for the repository (int8 artifact only; no training weights, no secrets)

In [ ]:
BUNDLE_OUT = RUNS / f"results-bundle-{EXPECTED_COMMIT[:12]}.tar"
args = [
    "ml.colab.bundle",
    "pack",
    "--out",
    str(BUNDLE_OUT),
    "--data",
    str(DATA),
    "--baselines",
    str(BASELINES),
    "--results",
    "ml/eval/results",
    "--artifact",
    str(ARTIFACT),
]
for name in ("deberta_v3_base", "modernbert_base", "deberta_v3_base_unseen4"):
    args += ["--run", f"{name}={RUNS / name}"]
run(*args)
crit = json.loads(pathlib.Path("ml/eval/results/baselines_comparison.json").read_text())[
    "handoff_criterion"
]
print("handoff criterion (model vs TF-IDF macro-F1 on TEST):", crit)
backup_to_drive(BUNDLE_OUT, BUNDLE_OUT.stem)
backup_to_drive(ARTIFACT, f"{ARTIFACT.name}-{EXPECTED_COMMIT[:12]}")
print(
    "download",
    BUNDLE_OUT,
    "(file browser), then run locally: make import-colab-bundle BUNDLE=<path>",
)